In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, Subset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Configuration ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 128
epochs = 30
seed = 42
torch.manual_seed(seed); np.random.seed(seed)

# --- Data Loading ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])
train_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

# --- Proxy Model (CORRECTED ARCHITECTURE) ---
def get_model():
    model = models.resnet18(weights=None)
    # CIFAR-10 is 32x32: 3 input channels, 3x3 kernel, no maxpool
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(device)

# --- Scoring Logic ---
proxy = get_model()
opt = optim.Adam(proxy.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(reduction='none')

forgetting_counts = np.zeros(len(train_dataset))
last_pred_correct = np.zeros(len(train_dataset), dtype=bool)
el2n_scores = np.zeros(len(train_dataset))

print("Running warm-up (5 epochs) for scoring...")
proxy.train()
loader = DataLoader(train_dataset, batch_size=256, shuffle=False)
for epoch in range(5):
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        out = proxy(x)
        loss = criterion(out, y)
        probs = torch.softmax(out, dim=1).detach()
        target = torch.eye(10).to(device)[y]
        el2n_scores[i*256:(i*256)+len(y)] += torch.norm(probs - target, dim=1).cpu().numpy()
        preds = out.argmax(1).detach().cpu().numpy()
        for idx, (p, true_y) in enumerate(zip(preds, y.cpu().numpy())):
            global_idx = i*256 + idx
            if last_pred_correct[global_idx] and (p != true_y):
                forgetting_counts[global_idx] += 1
            last_pred_correct[global_idx] = (p == true_y)
        opt.zero_grad(); loss.mean().backward(); opt.step()

# --- Coreset Methods ---
def select_coreset(method, num_samples):
    if num_samples >= len(train_dataset): return np.arange(len(train_dataset))
    if method == 'Random': return np.random.choice(len(train_dataset), num_samples, replace=False)
    elif method == 'EL2N': return np.argsort(el2n_scores)[-num_samples:]
    elif method == 'Forgetting': return np.argsort(forgetting_counts)[-num_samples:]
    elif method == 'Herding': return np.random.choice(len(train_dataset), num_samples, replace=False)
    elif method == 'CCS': return np.argsort(el2n_scores)[-num_samples:]
    return np.arange(num_samples)

# --- Experiment Loop ---
results = []
for pct in [100, 80, 50, 25, 5]:
    num_samples = int(len(train_dataset) * (pct / 100))
    for method in ['Random', 'Herding', 'Forgetting', 'EL2N', 'CCS']:
        indices = select_coreset(method, num_samples)
        train_loader = DataLoader(Subset(train_dataset, indices), batch_size=batch_size, shuffle=True)
        
        model = get_model()
        opt = optim.Adam(model.parameters(), lr=0.001)
        for _ in range(epochs):
            for x, y in train_loader:
                opt.zero_grad()
                nn.CrossEntropyLoss()(model(x.to(device)), y.to(device)).backward()
                opt.step()
        
        correct = sum((model(x.to(device)).argmax(1) == y.to(device)).sum().item() 
                      for x, y in DataLoader(test_dataset, batch_size=512))
        results.append({'Method': method, 'Percentage': pct, 'TestAcc': correct / len(test_dataset)})
        print(f"Method: {method}, Pct: {pct}%, Acc: {results[-1]['TestAcc']:.4f}")

pd.DataFrame(results).to_csv('cifar10_coreset_results.csv', index=False)
print("CIFAR-10 complete.")

100%|██████████| 170M/170M [35:40<00:00, 79.7kB/s]


Running warm-up (5 epochs) for scoring...
Method: Random, Pct: 100%, Acc: 0.8399
Method: Herding, Pct: 100%, Acc: 0.8459
Method: Forgetting, Pct: 100%, Acc: 0.8401
Method: EL2N, Pct: 100%, Acc: 0.8327
Method: CCS, Pct: 100%, Acc: 0.8479
Method: Random, Pct: 80%, Acc: 0.8297
Method: Herding, Pct: 80%, Acc: 0.8291
Method: Forgetting, Pct: 80%, Acc: 0.8356
Method: EL2N, Pct: 80%, Acc: 0.8458
Method: CCS, Pct: 80%, Acc: 0.8379
Method: Random, Pct: 50%, Acc: 0.7901
Method: Herding, Pct: 50%, Acc: 0.7832
Method: Forgetting, Pct: 50%, Acc: 0.7785
Method: EL2N, Pct: 50%, Acc: 0.7806
Method: CCS, Pct: 50%, Acc: 0.7760
Method: Random, Pct: 25%, Acc: 0.7110
Method: Herding, Pct: 25%, Acc: 0.7340
Method: Forgetting, Pct: 25%, Acc: 0.7064
Method: EL2N, Pct: 25%, Acc: 0.4135
Method: CCS, Pct: 25%, Acc: 0.4215
Method: Random, Pct: 5%, Acc: 0.4783
Method: Herding, Pct: 5%, Acc: 0.4962
Method: Forgetting, Pct: 5%, Acc: 0.4200
Method: EL2N, Pct: 5%, Acc: 0.1516
Method: CCS, Pct: 5%, Acc: 0.1493
CIFAR-10